# An emerging Allee effect

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/allee_effect.ipynb)

*Böttger K, Hatzikirou H, Voss-Böhme A, Cavalcanti-Adam EA, Herrero MA,
Deutsch A (2015). An emerging Allee effect is critical for tumor initiation
and persistence. PLoS Comput Biol 11(9): e1004366.
[doi:10.1371/journal.pcbi.1004366](https://doi.org/10.1371/journal.pcbi.1004366)*

**Question.** Tumour cells switch between migrating and dividing: the
go-or-grow dichotomy. If the switch depends on the local cell density, can
a small population die out although every cell is able to divide?

An Allee effect is a per-capita growth rate that is negative at small
population sizes, so that a population needs a minimum size to grow. In
ecology it comes from mating or cooperation; here it emerges from the
switch alone. This notebook reproduces the extinction frequency against the
initial density (Fig 3 of the paper), the bimodal outcomes near the
threshold, and the mean-field growth rate (Fig 5), and shows why the
lattice model's threshold lies well below the mean-field one.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.model import run_model
from lgca.study import sweep
from lgca.zoo import allee_effect as allee

zoo.parameter_table("allee_effect")

## The model

Cells live on a square lattice with $b = 4$ velocity channels and 4 rest
channels, $K = 8$ channels in all, with volume exclusion. Moving cells sit
in velocity channels, resting cells in rest channels. In every time step:

1. **R1, death:** every cell dies with probability $r_d$.
2. **R2, division:** every resting cell divides with probability $r_b$ if
   a rest channel is free; the daughter rests.
3. **R3, switch:** a moving cell starts resting with probability
   $r_s(\varrho)$, a resting cell starts moving with $1 - r_s(\varrho)$,
   where $\varrho = n / K$ is the density of its node and

   $$r_s(\varrho) = \tfrac12\left(1 + \tanh\big(\kappa(\varrho - \theta)\big)\right).$$

4. **R4, migration:** moving cells pick a random velocity channel, then
   move to the neighbouring node in that direction.

For $\kappa > 0$ cells rest where it is crowded and move where it is
sparse ("attraction"); for $\kappa < 0$ the opposite ("repulsion"). The
spec lists the rules in this order:

In [ ]:
spec = allee.build_spec()
for operator in spec.dynamics.operators:
    print(operator)
print(spec.space, spec.state.density, spec.time.steps)

`build_spec` starts from a uniform random density $\varrho_0$: every
channel is occupied with probability $\varrho_0$. The defaults are the
parameters of Fig 3 ($\kappa = 4.4$, $\theta = 0.75$) on a 50 × 50 lattice
for 1000 steps; `build_spec(full=True)` gives the paper's 100 × 100 lattice
and 5000 steps.

## Same start, two fates

Near the threshold the outcome is left to chance. Twelve runs of 4000 steps
from the same initial density $\varrho_0 = 0.26$, differing only in their
seeds (`sweep` runs them in parallel and, with `long=True`, returns the
population at every step):

In [ ]:
capacity = 50 * 50 * allee.K
fates = sweep(allee.build_spec(density=0.26, steps=4000), seeds=range(12), measure={"population": "population"},
              long=True, n_jobs=4, showprogress=False)
fates["density"] = fates["population"] / capacity
final = fates[fates["step"] == 4000].set_index("seed")["density"]

fig, axis = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
for seed, run in fates.groupby("seed"):
    axis.plot(run["step"], run["density"], color="tab:red" if final[seed] > 0.26 else "tab:blue", lw=1)
axis.set(xlabel="time step $k$", ylabel="density", title="12 seeds from $\\varrho_0 = 0.26$")
plt.show()
plt.close(fig)
print("densities after 4000 steps:", np.round(np.sort(final.values), 2))

The outcome is bimodal: after 4000 steps every run has either died out
or filled the lattice up to its steady density of about 0.86, never
anything in between, although some runs linger near the threshold for a
thousand steps before they take off. Snapshots of one run of each kind:

In [ ]:
dies, grows = int(final.idxmin()), int(final.idxmax())
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), constrained_layout=True)
for row, (seed, label) in enumerate(((dies, "dies out"), (grows, "grows"))):
    run = run_model(allee.build_spec(density=0.26, seed=seed, steps=400), showprogress=False)
    for column, step in enumerate((100, 400)):
        axis = axes[2 * row + column]
        frame = list(run.data.steps("density")).index(step)
        run.lgca.plot_density(density=run.data["density"][frame], ax=axis, vmax=allee.K,
                              cbar=axis is axes[-1], tight_layout=False)
        axis.set_title(f"seed {seed} ({label}), $k = {step}$")
plt.show()
plt.close(fig)

Some runs thin out and die, others fill the lattice. The declining run
shows the mechanism: in sparse regions almost every cell moves
($r_s(\varrho) \approx 0$ for $\varrho \ll \theta$), so almost none
divides, while death goes on. Where chance has gathered enough cells, they
rest, divide and raise the density further: a positive feedback that
either catches on or does not.

## Extinction frequency (Fig 3)

A sweep over the initial density with twelve seeds each and 1000 steps. A
run counts as declined if it ends with fewer cells than it started with;
most declined runs are extinct by then, the others follow later:

In [ ]:
densities = [0.18, 0.2, 0.22, 0.24, 0.26, 0.28, 0.3, 0.32]
table = sweep(
    allee.build_spec(),
    grid=[{"state.density": allee.K * rho} for rho in densities],
    seeds=range(12),
    measure={"final": allee.final_density},
    n_jobs=4,
    showprogress=False,
)
table["initial"] = table["density"] / allee.K
table["extinct"] = table["final"] == 0
table["grown"] = table["final"] > table["initial"]
summary = table.groupby("initial")[["extinct", "grown"]].mean()
summary

In [ ]:
fig, axis = plt.subplots(figsize=(6, 3.8), constrained_layout=True)
axis.plot(summary.index, 1 - summary["grown"], "s-", label="declined (extinct or shrinking)")
axis.plot(summary.index, summary["extinct"], "o--", label="already extinct")
axis.set(xlabel="initial density $\\varrho_0$", ylabel="frequency over 12 seeds")
axis.legend()
plt.show()
plt.close(fig)

As in the paper, the frequency of extinction falls from one to zero over a
narrow range of initial densities, here between 0.24 and 0.28, and in this
range the outcome is bimodal, as in the runs above.

## The mean-field growth rate, and why the lattice's threshold is lower (Fig 5)

If switching is fast compared with division, a fraction $r_s(\varrho)$ of
the cells rests, and the per-capita growth rate is (Eq 2 of the paper)

$$\frac{F(\varrho)}{\varrho} = r_s(\varrho)\, r_b - r_d .$$

For $\kappa > 0$ it is negative at small densities: an Allee effect. Its
zero, the mean-field threshold, lies at $\varrho^* \approx 0.42$ for the
parameters of Fig 3, but the simulations switch from decline to growth
near $0.25$. The difference is the discreteness of the lattice: a node
holds $n$ cells, binomially distributed around $K\varrho$, and switches
with $r_s(n/K)$, not with $r_s$ of the mean. Because $r_s$ rises steeply
with $n$, the few crowded nodes grow fast enough to outweigh the many
sparse ones. Averaging the growth rate over the cells of binomially
occupied nodes (`allee.per_capita_growth_nodes`) puts the threshold at
$0.24$:

In [ ]:
from scipy.optimize import brentq

rho = np.linspace(0.005, 1, 200)
mean_field = brentq(allee.per_capita_growth, 0.05, 0.95)
lattice = brentq(allee.per_capita_growth_nodes, 0.05, 0.95)
observed = summary.index[np.argmax(summary["grown"].values >= 0.5)]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for axis in axes:
    for kappa, colour in ((-4.4, "tab:purple"), (1.1, "tab:brown"), (4.4, "tab:orange")):
        axis.plot(rho, allee.per_capita_growth(rho, kappa=kappa), color=colour, label=f"mean field, κ = {kappa}")
    axis.axhline(0, color="grey", lw=0.8)
    axis.set(xlabel="density $\\varrho$", ylabel="per-capita growth rate", xlim=(0, 1))
axes[0].set_title("(a) the three kinds of switch")
axes[0].legend(fontsize=8)
axes[1].plot(rho, allee.per_capita_growth_nodes(rho), "k--", label="κ = 4.4, averaged over nodes")
axes[1].axvline(observed, color="tab:green", lw=6, alpha=0.3, label=f"simulated threshold ≈ {observed:.2f}")
axes[1].set(ylim=(-0.012, 0.03), title="(b) close to zero, κ = 4.4")
axes[1].legend(fontsize=8, loc="upper left")
plt.show()
plt.close(fig)
print(f"thresholds: mean field {mean_field:.3f}, averaged over nodes {lattice:.3f}")

With repulsion ($\kappa < 0$, purple) the growth rate is positive at
small densities and every population grows. With weak attraction
($\kappa = 1.1$, brown) growth is slowed at small densities but stays
positive; with strong attraction ($\kappa = 4.4$, orange) it turns
negative, and the Allee effect appears.

## Differences from the paper

- **Size.** 50 × 50 nodes and 1000 steps instead of 100 × 100 and 5000;
  `build_spec(full=True)` gives the paper's. The outcome is decided within
  1000 steps, and the threshold hardly depends on the size.
- **Initial condition.** Fig 3 plots extinction against the "averaged cell
  density"; we start from a uniform random density $\varrho_0$ over the
  whole lattice. The paper's movies start from a disc of radius 10 with one
  moving and one resting cell per node.
- **The movies.** The captions of S1 and S2 Video give $\kappa = 1.1$,
  $\theta = 0.375$, where the same disc dies out in one run and grows in
  another. With Eq (1) as printed, $r_s(\varrho)\,r_b > r_d$ at every
  density for these values, so the mean field predicts growth, and in our
  runs (100 × 100 lattice, the disc of the movies, four seeds) the
  population always grew. The two fates above use the parameters of Fig 3
  instead.
- **Order of the rules.** As listed in the paper, R1 to R4. The go-or-grow
  rule of earlier biolgca versions (`get_lgca(interaction="go_or_grow")`)
  switches first, then applies death and division.

## Explore it

Press **Play** and move $\kappa$ across zero, or the initial density
across the threshold (the density slider counts cells per node, $K
\varrho_0$, and builds the model again):

In [ ]:
explorer = lgca.explore(
    allee.build_spec(density=0.26),
    {"kappa": (-5.0, 5.0), "theta": (0.0, 1.0), "state.density": (0.0, 4.0)},
    steps_per_frame=5,
)
explorer

## Things to try

1. Repulsion: rerun the sweep with $\kappa = -4.4$. Does any population
   die out?
2. Change $\theta$. How does the threshold of the lattice model move, and
   does `per_capita_growth_nodes` follow it?
3. Start from the disc of the paper's movies (radius 10, one moving and
   one resting cell per node; build the `nodes` array and pass it as
   `state.nodes`). What is the extinction frequency at $\kappa = 4.4$? How
   large must the disc be to survive?
4. Measure the per-capita growth rate in the simulations, e.g. from the
   population over the first 50 steps, and compare it with both curves.
5. Replace the node density $\varrho = n/K$ by the density of the node and
   its neighbours (the `density` cue with `"scope": "neighbourhood"` in
   `go_or_rest(probability=...)`). Does the threshold move towards the
   mean field?